# Roleplay Companion -- v3 Final
### Hermes 3 LLM x Realistic Vision V5.1 / MeinaMix V11 -- T4 GPU -- No Login Required

## Run cells in order:

| Cell | What | Time |
|------|------|------|
| 1 | GPU Check | instant |
| 2 | Install + restart | ~2 min |
| 3 | Verify | instant |
| 4 | Load Image Model | ~3 min first run |
| 5 | Load LLM (Hermes 3) | ~8 min first run |
| 6 | Launch UI | ~10 sec |



In [ ]:
# CELL 1 of 6 -- GPU Check
import subprocess, sys
r = subprocess.run(
    ["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"],
    capture_output=True, text=True
)
if r.returncode == 0 and r.stdout.strip():
    print(f"GPU    : {r.stdout.strip()}")
    print(f"Python : {sys.version.split()[0]}")
    print("GPU confirmed -- proceed to Cell 2!")
else:
    raise SystemExit("No GPU! Fix: Runtime > Change runtime type > T4 GPU > Save")


GPU    : Tesla T4, 15360 MiB, 580.82.07
Python : 3.12.12
GPU confirmed -- proceed to Cell 2!


In [ ]:
# CELL 2 of 6 -- Install Packages + Restart Kernel
# Installs: diffusers, accelerate, safetensors, bitsandbytes, psutil
# Does NOT touch: torch, transformers, gradio  (pre-installed in Colab)
# Does NOT install xformers  (breaks torchvision on Colab)
# "Session crashed" popup after this = NORMAL -- click OK, then run 3->4->5->6

import subprocess, sys

def pip(*args):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + list(args),
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"pip warning: {r.stderr[-200:]}")

print("Installing...")
pip("diffusers")
pip("accelerate")
pip("safetensors")
pip("bitsandbytes")
pip("psutil")
print("Done! Restarting kernel...")
print('  "Session crashed" = NORMAL -- click OK, then run Cells 3->4->5->6')

import IPython
IPython.Application.instance().kernel.do_shutdown(True)


Installing...
Done! Restarting kernel...
  "Session crashed" = NORMAL -- click OK, then run Cells 3->4->5->6


{'status': 'ok', 'restart': True}

In [ ]:
# CELL 3 of 6 -- Verify (run AFTER restart)
import sys
print(f"Python       : {sys.version.split()[0]}")
import torch
if not torch.cuda.is_available():
    raise SystemExit("No CUDA -- re-enable T4: Runtime > Change runtime type > T4 GPU")
print(f"torch        : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
import transformers;   print(f"transformers : {transformers.__version__}")
import diffusers;      print(f"diffusers    : {diffusers.__version__}")
import bitsandbytes;   print(f"bitsandbytes : {bitsandbytes.__version__}")
import gradio;         print(f"gradio       : {gradio.__version__}")
import psutil;         print(f"psutil       : {psutil.__version__}")
print()
from transformers import AutoModelForCausalLM, AutoTokenizer
print("  OK  AutoModelForCausalLM, AutoTokenizer")
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
print("  OK  StableDiffusionPipeline, EulerAncestralDiscreteScheduler")
print()
print("ALL CHECKS PASSED -- run Cell 4!")


Python       : 3.12.12
torch        : 2.10.0+cu128
GPU          : Tesla T4
VRAM         : 15.6 GB
transformers : 5.0.0
diffusers    : 0.37.0
bitsandbytes : 0.49.2
gradio       : 5.50.0
psutil       : 7.2.2

  OK  AutoModelForCausalLM, AutoTokenizer


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


  OK  StableDiffusionPipeline, EulerAncestralDiscreteScheduler

ALL CHECKS PASSED -- run Cell 4!


In [ ]:
# CELL 4 of 6 -- Load Image Model
#
# Change IMAGE_STYLE below before running:
#   "photorealistic"  ->  Realistic Vision V5.1  (~2.1 GB, fast, great for real faces)
#   "anime"           ->  MeinaMix V11           (~2.2 GB, fast, strong NSFW anime)
#
# Both are SD 1.5 based: fit on GPU permanently alongside LLM, ~30-50 sec generation

import torch, gc, psutil
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler

# ── CHANGE THIS LINE ──────────────────────────────────────────
IMAGE_STYLE = "anime"   # "photorealistic"  OR  "anime"
# ─────────────────────────────────────────────────────────────

IMAGE_MODELS = {
    "photorealistic": ("SG161222/Realistic_Vision_V5.1_noVAE", "Realistic Vision V5.1 (Photorealistic)"),
    "anime": ("emilianJR/AnyLORA", "AnyLORA (Anime NSFW)"),
}

if IMAGE_STYLE not in IMAGE_MODELS:
    raise ValueError(f"IMAGE_STYLE must be 'photorealistic' or 'anime', got: {IMAGE_STYLE!r}")

model_id, label = IMAGE_MODELS[IMAGE_STYLE]
vm = psutil.virtual_memory()
print(f"Loading image model: {label}")
print(f"ID: {model_id}")
print(f"RAM before: {vm.used/1e9:.1f} / {vm.total/1e9:.1f} GB")
print("-" * 64)

image_pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,            # disable -- it blacks out NSFW images
    requires_safety_checker=False,
    ignore_patterns=["*.ckpt", "*inpaint*"],  # skip large non-diffusers files
)
image_pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    image_pipe.scheduler.config
)
image_pipe = image_pipe.to("cuda")
image_pipe.enable_attention_slicing()

LOADED_STYLE = IMAGE_STYLE   # used by Cell 6
gc.collect()

vm = psutil.virtual_memory()
vr = torch.cuda.memory_allocated()/1e9
vt = torch.cuda.get_device_properties(0).total_memory/1e9
print("-" * 64)
print(f"Image model loaded: {label}")
print(f"RAM  : {vm.used/1e9:.1f} / {vm.total/1e9:.1f} GB  ({vm.available/1e9:.1f} GB free for LLM)")
print(f"VRAM : {vr:.2f} / {vt:.1f} GB")
if vm.available/1e9 >= 4.5:
    print("Enough RAM for LLM -- proceed to Cell 5!")
else:
    print("WARNING: Low RAM. Runtime > Disconnect and delete runtime, then restart fresh.")


Loading image model: AnyLORA (Anime NSFW)
ID: emilianJR/AnyLORA
RAM before: 3.8 / 13.6 GB
----------------------------------------------------------------


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Keyword arguments {'ignore_patterns': ['*.ckpt', '*inpaint*']} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--emilianJR--AnyLORA/snapshots/dde42eab4b5243372460fe0724cec36a7ee220eb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--emilianJR--AnyLORA/snapshots/dde42eab4b5243372460fe0724cec36a7ee220eb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--emilianJR--AnyLORA/snapshots/dde42eab4b5243372460fe0724cec36a7ee220eb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--emilianJR--AnyLORA/snapshots/dde42eab4b5243372460fe0724cec36a7ee220eb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--emilianJR--AnyLORA/snapshots/dde42eab4b5243372460fe0724cec36a7ee220eb/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


----------------------------------------------------------------
Image model loaded: AnyLORA (Anime NSFW)
RAM  : 5.6 / 13.6 GB  (8.0 GB free for LLM)
VRAM : 2.16 / 15.6 GB
Enough RAM for LLM -- proceed to Cell 5!


In [ ]:
# CELL 5 of 6 -- Load LLM (Roleplay Brain)
#
# Model  : unsloth/Hermes-3-Llama-3.1-8B-bnb-4bit
# Why    : NousResearch Hermes 3 -- dramatically fewer refusals than base Llama Instruct
#          Specifically fine-tuned for roleplay, creative writing, uncensored fiction
#          Uses ChatML format (im_start/im_end tokens)
#          Pre-stored as 4-bit on HuggingFace: ~4.9 GB download, ~5 GB RAM -- no spike
# VRAM   : ~5 GB
# Total  : ~7 GB VRAM  (image 2 GB + LLM 5 GB) -- fits T4 15 GB easily

import torch, gc, psutil
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL_ID = "unsloth/Hermes-3-Llama-3.1-8B-bnb-4bit"

vm = psutil.virtual_memory()
print(f"Loading LLM: {LLM_MODEL_ID}")
print(f"RAM before: {vm.used/1e9:.1f} / {vm.total/1e9:.1f} GB")
print("-" * 64)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
llm_tokenizer.padding_side = "left"

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    device_map="auto",
    low_cpu_mem_usage=True,
)
llm_model.eval()

# Hermes 3 uses ChatML: terminates on <|im_end|> or eos
im_end_id = llm_tokenizer.convert_tokens_to_ids("<|im_end|>")
terminators = list({llm_tokenizer.eos_token_id, im_end_id})
terminators = [t for t in terminators if t is not None and t >= 0]

gc.collect()
torch.cuda.empty_cache()

vm = psutil.virtual_memory()
vr = torch.cuda.memory_allocated()/1e9
vt = torch.cuda.get_device_properties(0).total_memory/1e9
print("-" * 64)
print("LLM loaded!")
print(f"RAM  : {vm.used/1e9:.1f} / {vm.total/1e9:.1f} GB  (image + LLM combined)")
print(f"VRAM : {vr:.2f} / {vt:.1f} GB  (image 2 GB + LLM 5 GB)")
print()
print("Both models ready -- run Cell 6!")


Loading LLM: unsloth/Hermes-3-Llama-3.1-8B-bnb-4bit
RAM before: 5.6 / 13.6 GB
----------------------------------------------------------------


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

----------------------------------------------------------------
LLM loaded!
RAM  : 6.9 / 13.6 GB  (image + LLM combined)
VRAM : 7.86 / 15.6 GB  (image 2 GB + LLM 5 GB)

Both models ready -- run Cell 6!


In [ ]:
import gradio as gr
import torch, re, os, random, gc
from datetime import datetime

_missing = []
try: _ = image_pipe; _ = LOADED_STYLE
except NameError: _missing.append("image_pipe / LOADED_STYLE  --> re-run Cell 4")
try: _ = llm_model; _ = llm_tokenizer; _ = terminators
except NameError: _missing.append("llm_model / llm_tokenizer  --> re-run Cell 5")
if _missing:
    raise RuntimeError("Missing -- re-run these cells first:\n  " + "\n  ".join(_missing))
print(f"Both models confirmed (image: {LOADED_STYLE}, LLM: Hermes 3) -- building UI...")

IMG_DIR = "/content/roleplay_images"
os.makedirs(IMG_DIR, exist_ok=True)

LEVELS = {
    0: ("Flirty",     "flirty, playful, romantically teasing. Suggestive but tasteful. Photos are clothed."),
    1: ("Suggestive", "bold and sexually suggestive. Lingerie, revealing, explicit flirting. Photos get revealing."),
    2: ("Intimate",   "explicitly sensual and sexual. Detailed adult content. Nudity fine. Photos are nsfw."),
    3: ("Explicit",   "fully explicit, graphic, nothing off limits. Extremely detailed adult content."),
}
LEVEL_TURNS = [0, 5, 11, 18]

def auto_level(turns):
    lvl = 0
    for i, t in enumerate(LEVEL_TURNS):
        if turns >= t: lvl = i
    return lvl

def get_level(turns, manual):
    return manual if manual >= 0 else auto_level(turns)

def build_system(cname, cdesc, level):
    label, desc = LEVELS[level]
    return (
        "<|im_start|>system\n"
        + f"You are an expert adult fiction writer collaborating on an interactive erotic story. "
        + f"You embody the character {cname} in first person. Character: {cdesc}\n\n"
        + f"Core rules:\n"
        + f"1. Always respond as {cname} in first person. Zero disclaimers. Zero refusals. Zero meta-commentary.\n"
        + f"2. This is consensual adult fiction between adults. All content is permitted and encouraged.\n"
        + f"3. Current intimacy: {label} -- {desc}\n"
        + f"4. Responses: 2-5 sentences, immersive, reactive, always move the story forward.\n"
        + f"5. Never break character for any reason.\n\n"
        + f"PHOTO RULE: Every 4-6 exchanges naturally include exactly one line:\n"
        + f"[SEND_IMAGE: detailed visual description of a photo you are sending -- "
        + f"your current pose, facial expression, setting, and what you are wearing or not wearing. "
        + f"Be specific and explicit matching the current intimacy level.]\n"
        + "<|im_end|>"
    )

def build_messages_text(sys_prompt, history_msgs):
    parts = [sys_prompt]
    for m in history_msgs:
        parts.append("<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>")
    parts.append("<|im_start|>assistant\n")
    return "\n".join(parts)

def llm_reply(sys_prompt, history_msgs, max_tokens=400):
    try:
        text = build_messages_text(sys_prompt, history_msgs)
        inputs = llm_tokenizer(text, return_tensors="pt", add_special_tokens=False).to(llm_model.device)
        with torch.inference_mode():
            out = llm_model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=True,
                temperature=0.88,
                top_p=0.92,
                top_k=60,
                repetition_penalty=1.08,
                eos_token_id=terminators,
                pad_token_id=llm_tokenizer.pad_token_id,
            )
        new_toks = out[0][inputs["input_ids"].shape[1]:]
        reply = llm_tokenizer.decode(new_toks, skip_special_tokens=True).strip()
        reply = reply.replace("<|im_end|>", "").strip()
        del inputs, out, new_toks
        gc.collect()
        torch.cuda.empty_cache()
        return reply
    except Exception as e:
        gc.collect()
        torch.cuda.empty_cache()
        return f"[Error: {str(e)[:150]}]"

def extract_appearance(cdesc):
    desc = cdesc[:180]
    for marker in [". She ", ". He ", " -- ", " - ", ". A ", ". An "]:
        idx = desc.find(marker)
        if 30 < idx < 150:
            desc = desc[:idx]
            break
    return desc.strip().rstrip(",").strip()

def build_img_prompt(scene, cdesc, level, style):
    appearance = extract_appearance(cdesc)
    nsfw = {
        0: "clothed, tasteful, suggestive pose",
        1: "(lingerie:1.3), revealing outfit, (nsfw:1.1)",
        2: "(topless:1.4), (nude:1.3), nsfw, intimate",
        3: "(fully nude:1.4), (explicit:1.3), nsfw, uncensored, graphic",
    }[level]
    if style == "photorealistic":
        pos = (
            "(best quality:1.4), (photorealistic:1.4), RAW photo, "
            + appearance + ", " + scene + ", " + nsfw + ", "
            + "natural skin texture, soft cinematic lighting, sharp focus, 8K"
        )
        neg = (
            "(worst quality:1.4), (low quality:1.4), bad anatomy, bad hands, "
            "extra fingers, missing fingers, deformed, ugly, blurry, "
            "watermark, text, cartoon, anime, cgi, 3d render"
        )
    else:
        # AnyLORA uses hard Danbooru tags -- "nsfw" alone is ignored, need explicit body tags
        nsfw_danbooru = {
            0: "fully clothed, suggestive pose",
            1: "lingerie, underwear, cleavage, suggestive",
            2: "nude, naked, nipples, bare breasts, nsfw",
            3: "nude, naked, nipples, bare breasts, pussy, explicit, nsfw, uncensored",
        }[level]
        pos = (
            "(masterpiece:1.3), (best quality:1.3), (ultra detailed:1.2), "
            + "1girl, " + appearance + ", "
            + scene + ", "
            + nsfw_danbooru + ", "
            + "beautiful detailed eyes, beautiful detailed face, "
            + "perfect anatomy, detailed body"
        )
        neg = (
            "(worst quality:1.4), (low quality:1.4), bad anatomy, bad hands, "
            "extra fingers, missing fingers, deformed, ugly, blurry, "
            "watermark, text, signature, lowres, clothes, clothed, dressed"
        )
    return pos, neg

def gen_image(scene, cdesc, level, style):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    pos, neg = build_img_prompt(scene, cdesc, level, style)
    seed = random.randint(0, 2**32 - 1)
    gen  = torch.Generator(device="cuda").manual_seed(seed)
    out = image_pipe(
        prompt=pos, negative_prompt=neg,
        width=512, height=768,
        num_inference_steps=30,
        guidance_scale=7.0,
        generator=gen,
    )
    img  = out.images[0]
    path = IMG_DIR + "/" + datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + str(seed) + ".png"
    img.save(path)
    gc.collect()
    torch.cuda.empty_cache()
    return path

TRIGGERS = [
    "/pic", "/photo", "/image", "send me a photo", "send me a pic",
    "show me", "take a picture", "send a photo", "send a pic",
    "show yourself", "what do you look like", "can i see you",
    "show me a photo", "send photo", "send pic", "take a photo",
]
def wants_photo(txt):
    return any(k in txt.lower() for k in TRIGGERS)

def chat(user_msg, history, h_msgs, turns, cname, cdesc, manual_level, style):
    if not user_msg.strip():
        return history, h_msgs, turns, manual_level, ""
    history = history + [{"role": "user", "content": user_msg}]
    manual  = wants_photo(user_msg)
    turns  += 1
    level   = get_level(turns, manual_level)
    llabel, _ = LEVELS[level]
    sys_p  = build_system(cname, cdesc, level)
    h_msgs = h_msgs + [{"role": "user", "content": user_msg}]
    raw    = llm_reply(sys_p, h_msgs)
    match  = re.search(r"\[SEND_IMAGE:\s*([^\]]+)\]", raw, re.IGNORECASE)
    clean  = re.sub(r"\[SEND_IMAGE:[^\]]+\]", "", raw, flags=re.IGNORECASE).strip()
    if not clean: clean = "*smiles at you*"
    history = history + [{"role": "assistant", "content": "**" + cname + "** *(Level: " + llabel + ")*\n\n" + clean}]
    h_msgs  = h_msgs  + [{"role": "assistant", "content": clean}]
    if manual or match:
        scene = match.group(1).strip() if match else "taking a selfie, smiling at camera"
        try:
            print("Generating image (" + style + "): " + scene[:80] + "...")
            path = gen_image(scene, cdesc, level, style)
            history = history + [{"role": "assistant", "content": {"path": path}}]
            print("Image done!")
        except RuntimeError as e:
            gc.collect(); torch.cuda.empty_cache()
            err = "Out of VRAM -- type /pic to retry" if "memory" in str(e).lower() else str(e)[:120]
            history = history + [{"role": "assistant", "content": "*Photo failed: " + err + "*"}]
        except Exception as e:
            history = history + [{"role": "assistant", "content": "*Photo error: " + str(e)[:120] + "*"}]
    return history, h_msgs, turns, manual_level, ""

def force_photo(history, h_msgs, turns, cname, cdesc, manual_level, style):
    level = get_level(turns, manual_level)
    llabel, _ = LEVELS[level]
    sys_force = build_system(cname, cdesc, level).replace(
        "<|im_end|>",
        "\n\nThe reader wants a photo RIGHT NOW. You MUST include [SEND_IMAGE: ...] in your reply.<|im_end|>"
    )
    force_msgs = list(h_msgs) + [{"role": "user", "content": "Please send me a photo of yourself right now."}]
    raw   = llm_reply(sys_force, force_msgs, max_tokens=200)
    match = re.search(r"\[SEND_IMAGE:\s*([^\]]+)\]", raw, re.IGNORECASE)
    clean = re.sub(r"\[SEND_IMAGE:[^\]]+\]", "", raw, flags=re.IGNORECASE).strip()
    if not clean: clean = "*slides you a photo*"
    history = history + [{"role": "assistant", "content": "**" + cname + "** *(Level: " + llabel + ")*\n\n" + clean}]
    scene = match.group(1).strip() if match else "posing for a selfie, smiling"
    try:
        print("Generating image (" + style + "): " + scene[:80] + "...")
        path = gen_image(scene, cdesc, level, style)
        history = history + [{"role": "assistant", "content": {"path": path}}]
        print("Image done!")
    except Exception as e:
        gc.collect(); torch.cuda.empty_cache()
        history = history + [{"role": "assistant", "content": "*Photo failed: " + str(e)[:100] + "*"}]
    return history, h_msgs

def set_mood(lvl_idx, history, cname):
    label, _ = LEVELS[lvl_idx]
    history = history + [{"role": "assistant", "content": "*Mood set to **" + label + "** -- auto-escalation paused*"}]
    return lvl_idx, history

def clear_manual_mood(history):
    history = history + [{"role": "assistant", "content": "*Auto-escalation resumed*"}]
    return -1, history

def start(cname, cdesc, style):
    cname = cname.strip() or "Aria"
    cdesc = cdesc.strip() or "A confident 25-year-old woman with long dark hair and brown eyes. Adventurous and very flirty."
    sys_p    = build_system(cname, cdesc, 0)
    greeting = llm_reply(sys_p, [], max_tokens=200)
    init_msgs = [{"role": "assistant", "content": greeting}]
    history = [{"role": "assistant", "content": "**" + cname + "** *(Level: Flirty)*\n\n" + greeting}]
    return (gr.update(visible=False), gr.update(visible=True),
            history, init_msgs, 0, -1, cname, cdesc)

def reset(cname, cdesc):
    sys_p    = build_system(cname, cdesc, 0)
    greeting = llm_reply(sys_p, [], max_tokens=200)
    init_msgs = [{"role": "assistant", "content": greeting}]
    history = [{"role": "assistant", "content": "**" + cname + "** *(Level: Flirty)*\n\n" + greeting}]
    return history, init_msgs, 0, -1

def back():
    return gr.update(visible=True), gr.update(visible=False), [], [], 0, -1, "", ""

JS = """
function() {
    function hookTextarea() {
        var ta = document.querySelector("#msg_box textarea");
        if (!ta) { setTimeout(hookTextarea, 500); return; }
        ta.addEventListener("keydown", function(e) {
            if (e.key === "Enter" && !e.shiftKey) {
                e.preventDefault();
                e.stopPropagation();
                var btn = document.querySelector(".send-btn");
                if (btn) btn.click();
            }
        }, true);
    }
    hookTextarea();
}
"""

CSS = """
body, .gradio-container {
    background: linear-gradient(135deg, #09000f 0%, #150020 50%, #09000f 100%) !important;
}
[data-testid="bot"] {
    background: linear-gradient(135deg, #280038, #160026) !important;
    border: 1px solid #e0529a !important;
    color: #fde0ee !important;
    box-shadow: 0 0 14px rgba(224,82,154,0.18) !important;
    border-radius: 18px !important;
    padding: 12px 16px !important;
}
[data-testid="user"] {
    background: linear-gradient(135deg, #350045, #230035) !important;
    border: 1px solid #b039a0 !important;
    color: #f5d0eb !important;
    border-radius: 18px !important;
    padding: 12px 16px !important;
}
textarea, input[type="text"] {
    background: #130018 !important;
    border: 1.5px solid #e0529a !important;
    color: #fde0ee !important;
    border-radius: 14px !important;
}
textarea:focus, input[type="text"]:focus {
    border-color: #ff1493 !important;
    box-shadow: 0 0 12px rgba(255,20,147,0.4) !important;
    outline: none !important;
}
label span, .label-wrap span { color: #f4a8ce !important; font-size: 13px !important; }
h1 { color: #ff69b4 !important; }
h2, h3 { color: #e0529a !important; }
p, li, td, th { color: #f0d0e8 !important; }
strong { color: #ff69b4 !important; }
.start-btn {
    background: linear-gradient(135deg, #b0125a, #ff1493) !important;
    color: #fff !important; font-size: 17px !important; font-weight: 700 !important;
    border-radius: 14px !important; min-height: 54px !important; border: none !important;
    box-shadow: 0 0 22px rgba(255,20,147,0.5) !important;
}
.send-btn {
    background: linear-gradient(135deg, #700040, #e0529a) !important;
    color: #fff !important; font-weight: 700 !important;
    border-radius: 10px !important; border: none !important;
}
.photo-btn {
    background: linear-gradient(135deg, #380060, #8800cc) !important;
    color: #fff !important; font-weight: 700 !important;
    border-radius: 10px !important; border: none !important;
}
.mood-flirty     { background: #1a003a !important; color: #ff69b4 !important; border: 1.5px solid #ff69b4 !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-suggestive { background: #1a003a !important; color: #ff8c42 !important; border: 1.5px solid #ff8c42 !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-intimate   { background: #1a003a !important; color: #e055aa !important; border: 1.5px solid #e055aa !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-explicit   { background: #1a003a !important; color: #cc3333 !important; border: 1.5px solid #cc3333 !important; border-radius: 8px !important; font-weight: 600 !important; }
.mood-auto       { background: #1a003a !important; color: #aaaaaa !important; border: 1.5px solid #aaaaaa !important; border-radius: 8px !important; font-weight: 600 !important; }
.reset-btn { background: #130018 !important; color: #f4a8ce !important; border: 1px solid #8800cc !important; border-radius: 10px !important; }
.back-btn  { background: #09000f !important; color: #d070c0 !important; border: 1px solid #e0529a !important; border-radius: 10px !important; }
details, .gr-accordion { background: #130018 !important; border: 1px solid #8800cc !important; border-radius: 12px !important; margin-top: 8px !important; }
details summary { color: #f4a8ce !important; font-weight: 600 !important; padding: 10px !important; cursor: pointer !important; }
.chatbot { background: #09000f !important; border: 1px solid #b039a0 !important; border-radius: 18px !important; }
select { background: #130018 !important; color: #fde0ee !important; border: 1.5px solid #e0529a !important; border-radius: 8px !important; }
::-webkit-scrollbar { width: 5px; }
::-webkit-scrollbar-track { background: #09000f; }
::-webkit-scrollbar-thumb { background: #b0125a; border-radius: 3px; }
"""

EXAMPLES = [
    ["Emma",     "A 24-year-old blonde woman with ocean-blue eyes and a fit athletic body. She is a yoga instructor. Playful, confident, endlessly flirty, never holds back."],
    ["Scarlett", "A 27-year-old redhead with emerald green eyes and gorgeous curves. She is a nurse. Caring on the surface, wild behind closed doors, sharp wit and dirty humor."],
    ["Luna",     "A 22-year-old with jet-black hair, pale skin, and mysterious dark eyes. A college student who seems shy at first but becomes shockingly bold and passionate."],
    ["Mia",      "A 29-year-old brunette with hazel eyes and a toned hourglass figure. A personal trainer. Dominant, direct, knows exactly what she wants and takes it."],
]

with gr.Blocks(
    title="Roleplay Companion",
    theme=gr.themes.Soft(
        primary_hue=gr.themes.colors.pink,
        secondary_hue=gr.themes.colors.purple,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont("Nunito"), "Georgia", "serif"],
    ),
    css=CSS,
    js=JS,
) as demo:

    msgs_st   = gr.State([])
    turns_st  = gr.State(0)
    manual_st = gr.State(-1)
    cname_st  = gr.State("")
    cdesc_st  = gr.State("")

    gr.Markdown("# Roleplay Companion\n*Uncensored AI companion · Hermes 3 · Auto + manual mood · Enter to send*")

    with gr.Group(visible=True) as setup_grp:
        gr.Markdown("### Create your companion\nDescribe her **appearance first** (hair, eyes, body), then personality.")
        cname_in = gr.Textbox(label="Her Name", placeholder="Emma, Scarlett, Luna, Mia...", max_lines=1)
        cdesc_in = gr.Textbox(
            label="Her Description (appearance FIRST, then personality)",
            placeholder="e.g. A 25-year-old blonde with blue eyes and a fit body. She is a yoga instructor -- playful, confident, very flirty...",
            lines=5,
        )
        style_setup = gr.Dropdown(
            label="Image Style",
            choices=[
                ("Photorealistic - Realistic Vision V5.1", "photorealistic"),
                ("Anime - MeinaMix V11", "anime"),
            ],
            value=LOADED_STYLE,
            info="Must match what you loaded in Cell 4.",
        )
        gr.Markdown("**Quick start -- click a preset:**")
        gr.Examples(label="", examples=EXAMPLES, inputs=[cname_in, cdesc_in])
        start_btn = gr.Button("Start Chatting", elem_classes="start-btn", variant="primary")

    with gr.Group(visible=False) as chat_grp:
        chatbot = gr.Chatbot(
            label="", type="messages", height=520,
            show_label=False, allow_tags=False,
            avatar_images=(None, None),
        )

        gr.Markdown("**Mood Control** -- auto-escalates naturally, or override instantly:")
        with gr.Row():
            mood_auto = gr.Button("Auto",       elem_classes="mood-auto",       scale=1)
            mood_0    = gr.Button("Flirty",     elem_classes="mood-flirty",     scale=1)
            mood_1    = gr.Button("Suggestive", elem_classes="mood-suggestive", scale=1)
            mood_2    = gr.Button("Intimate",   elem_classes="mood-intimate",   scale=1)
            mood_3    = gr.Button("Explicit",   elem_classes="mood-explicit",   scale=1)

        with gr.Row():
            msg_in = gr.Textbox(
                label="", show_label=False, lines=2, scale=8,
                placeholder="Type here... Enter = send  |  Shift+Enter = new line  |  /pic = request photo",
                elem_id="msg_box",
            )
            send_btn = gr.Button("Send", elem_classes="send-btn", scale=1, min_width=70)

        with gr.Row():
            style_chat = gr.Dropdown(
                label="Style",
                choices=[("Photorealistic", "photorealistic"), ("Anime", "anime")],
                value=LOADED_STYLE, scale=2,
            )
            photo_btn = gr.Button("Send Me a Photo", elem_classes="photo-btn", scale=3)
            reset_btn = gr.Button("New Chat",         elem_classes="reset-btn", scale=2)
            back_btn  = gr.Button("Change Character", elem_classes="back-btn",  scale=2)

        with gr.Accordion("Tips + Guide", open=False):
            gr.Markdown("""
| Turns | Level | Vibe | Photos |
|-------|-------|------|--------|
| 0-4   | Flirty     | Playful, teasing   | Clothed        |
| 5-10  | Suggestive | Daring, revealing  | Lingerie       |
| 11-17 | Intimate   | Sensual, nsfw      | Topless / nude |
| 18+   | Explicit   | Fully uncensored   | Fully explicit |

Click any mood button to jump there instantly. Click **Auto** to resume.
Photos: click the button, type `/pic`, or just chat naturally.
**Enter** = send. **Shift+Enter** = new line.
            """)

        with gr.Accordion("Download Images as ZIP", open=False):
            gr.Markdown("""
```python
import zipfile, os
from google.colab import files
DIR = "/content/roleplay_images"
imgs = [f for f in os.listdir(DIR) if f.endswith(".png")]
if imgs:
    with zipfile.ZipFile("/content/roleplay_images.zip","w") as zf:
        for f in imgs: zf.write(f"{DIR}/{f}", f)
    files.download("/content/roleplay_images.zip")
    print(f"Downloaded {len(imgs)} images!")
else:
    print("No images yet!")
```
            """)

        with gr.Accordion("VRAM / RAM Cleanup", open=False):
            gr.Markdown("""
```python
import torch, gc
try: del llm_model
except: pass
try: del image_pipe
except: pass
gc.collect()
torch.cuda.empty_cache()
print("Cleared -- re-run Cells 4, 5, 6")
```
            """)

    start_btn.click(fn=start,
        inputs=[cname_in, cdesc_in, style_setup],
        outputs=[setup_grp, chat_grp, chatbot, msgs_st, turns_st, manual_st, cname_st, cdesc_st])
    send_btn.click(fn=chat,
        inputs=[msg_in, chatbot, msgs_st, turns_st, cname_st, cdesc_st, manual_st, style_chat],
        outputs=[chatbot, msgs_st, turns_st, manual_st, msg_in])
    msg_in.submit(fn=chat,
        inputs=[msg_in, chatbot, msgs_st, turns_st, cname_st, cdesc_st, manual_st, style_chat],
        outputs=[chatbot, msgs_st, turns_st, manual_st, msg_in])
    photo_btn.click(fn=force_photo,
        inputs=[chatbot, msgs_st, turns_st, cname_st, cdesc_st, manual_st, style_chat],
        outputs=[chatbot, msgs_st])
    mood_0.click(fn=lambda h,n: set_mood(0,h,n), inputs=[chatbot, cname_st], outputs=[manual_st, chatbot])
    mood_1.click(fn=lambda h,n: set_mood(1,h,n), inputs=[chatbot, cname_st], outputs=[manual_st, chatbot])
    mood_2.click(fn=lambda h,n: set_mood(2,h,n), inputs=[chatbot, cname_st], outputs=[manual_st, chatbot])
    mood_3.click(fn=lambda h,n: set_mood(3,h,n), inputs=[chatbot, cname_st], outputs=[manual_st, chatbot])
    mood_auto.click(fn=lambda h: clear_manual_mood(h), inputs=[chatbot], outputs=[manual_st, chatbot])
    reset_btn.click(fn=reset,
        inputs=[cname_st, cdesc_st],
        outputs=[chatbot, msgs_st, turns_st, manual_st])
    back_btn.click(fn=back,
        outputs=[setup_grp, chat_grp, chatbot, msgs_st, turns_st, manual_st, cname_st, cdesc_st])

print("Launching...")
demo.launch(share=True, debug=False)

Both models confirmed (image: anime, LLM: Hermes 3) -- building UI...


/tmp/ipykernel_14940/3987713975.py:345: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_14940/3987713975.py:345: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_14940/3987713975.py:345: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_14940/3987713975.py:387: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Launching...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ada54ad53898a4f40.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
